# EPI: novo split por cena e retreino
Agrupa imagens parecidas (hash perceptual), divide treino, validação e teste por grupo e retreina. Assim o teste só tem cenas que o modelo nunca viu.

## 1. Setup

In [ ]:
!pip -q install -U ultralytics imagehash

In [ ]:
import os, shutil, random, yaml
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import imagehash
from PIL import Image
from IPython.display import display
from scipy.sparse.csgraph import connected_components
from ultralytics import YOLO

os.environ['WANDB_MODE'] = 'disabled'
BASE = Path('/kaggle/working')
RUNS = BASE / 'runs'
NAMES = ['helmet', 'no_helmet', 'no_vest', 'person', 'vest']
SPLITS = ['train', 'valid', 'test']
DS = next(Path('/kaggle/input').rglob('data.yaml')).parent
print(DS)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SEM GPU')

## 2. Hash perceptual de todas as imagens

In [ ]:
itens = [(s, p) for s in SPLITS for p in sorted((DS / s / 'images').glob('*'))]
H = np.array([imagehash.phash(Image.open(p).convert('RGB'), hash_size=8).hash.flatten()
              for _, p in itens], dtype=np.float32)
soma = H.sum(1)
D = (soma[:, None] + soma[None, :] - 2 * H @ H.T).astype(np.int16)
print(len(itens), 'imagens')

## 3. Grupos por cena
Imagens com distância de hash menor ou igual a `LIMIAR` ficam no mesmo grupo (componentes conectados).

In [ ]:
LIMIAR = 10
_, grupo = connected_components(D <= LIMIAR, directed=False)
tam = pd.Series(grupo).value_counts()
print('grupos:', len(tam))
print('maior grupo:', tam.iloc[0], 'imagens')
print('grupos com 1 imagem:', int((tam == 1).sum()))
print(tam.head(10).to_string())
tam.clip(upper=30).hist(bins=30, figsize=(7, 3))
plt.title('Imagens por grupo (cortado em 30)')
plt.show()

## 4. Novo split por grupo
Grupos maiores que 10% do dataset vão para o treino. Confira a tabela de classes por split antes de seguir.

In [ ]:
rng = random.Random(42)
ids = list(tam.index)
rng.shuffle(ids)

alvo = 0.1 * len(itens)
cont = {'train': 0, 'valid': 0, 'test': 0}
destino = {}
for g in ids:
    n = int(tam[g])
    if n > alvo:
        d = 'train'
    elif cont['test'] < alvo:
        d = 'test'
    elif cont['valid'] < alvo:
        d = 'valid'
    else:
        d = 'train'
    destino[g] = d
    cont[d] += n

def classes_da_imagem(s, p):
    lab = DS / s / 'labels' / (p.stem + '.txt')
    if not lab.exists():
        return []
    return [int(l.split()[0]) for l in lab.read_text().splitlines() if l.strip()]

meta = pd.DataFrame({
    'arquivo': [f'{s}_{p.stem}{p.suffix}' for s, p in itens],
    'split_original': [s for s, _ in itens],
    'grupo': grupo,
})
meta['split'] = meta.grupo.map(destino)
meta['classes'] = [classes_da_imagem(s, p) for s, p in itens]

display(meta.split.value_counts().to_frame('imagens'))
display(meta.groupby('split').grupo.nunique().to_frame('grupos'))

inst = meta[['split', 'classes']].explode('classes').dropna()
tab = pd.crosstab(inst.classes.map(dict(enumerate(NAMES))), inst.split)[SPLITS]
display(tab)

In [ ]:
ix = {s: np.where(meta.split == s)[0] for s in SPLITS}
for a, b in [('test', 'train'), ('valid', 'train'), ('test', 'valid')]:
    d = D[np.ix_(ix[a], ix[b])].min(1)
    print(a, 'vs', b, '| distância mínima:', d.min(), '| imagens com vizinho <= 10:', int((d <= 10).sum()))

## 5. Montar o dataset

In [ ]:
OUT = Path('/kaggle/temp/ppe_grupos')
shutil.rmtree(OUT, ignore_errors=True)
for s in SPLITS:
    (OUT / s / 'images').mkdir(parents=True)
    (OUT / s / 'labels').mkdir(parents=True)

for (s0, p), s, arq in zip(itens, meta.split, meta.arquivo):
    shutil.copy(p, OUT / s / 'images' / arq)
    lab = DS / s0 / 'labels' / (p.stem + '.txt')
    texto = lab.read_text() if lab.exists() else ''
    (OUT / s / 'labels' / (Path(arq).stem + '.txt')).write_text(texto)

yaml.safe_dump(
    {'path': str(OUT), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
     'nc': len(NAMES), 'names': NAMES},
    open(OUT / 'data.yaml', 'w'))

DATA = str(OUT / 'data.yaml')
print({s: len(list((OUT / s / 'images').glob('*'))) for s in SPLITS})

## 6. Treino
Hiperparâmetros padrão (o tuning anterior não achou nada melhor). YOLO26s e YOLO26n, para comparar precisão e velocidade.

In [ ]:
MODELOS = ['yolo26s.pt', 'yolo26n.pt']
EPOCHS = 80
resultados = []
pesos = {}

for m in MODELOS:
    nome = m.replace('.pt', '')
    modelo = YOLO(m)
    modelo.train(data=DATA, epochs=EPOCHS, imgsz=640, batch=16, patience=20, seed=42,
                 project=str(RUNS), name='grupo_' + nome, exist_ok=True)
    pesos[nome] = modelo.trainer.best
    for split in ['val', 'test']:
        r = YOLO(pesos[nome]).val(data=DATA, split=split, imgsz=640, plots=False)
        resultados.append({'modelo': nome, 'split': split, 'precisao': r.box.mp, 'recall': r.box.mr,
                           'mAP50': r.box.map50, 'mAP50-95': r.box.map, 'ms_por_imagem': r.speed['inference']})

tabela = pd.DataFrame(resultados)
display(tabela.round(4))

## 7. Avaliação por classe no teste

In [ ]:
def por_classe(peso, tag):
    r = YOLO(peso).val(data=DATA, split='test', imgsz=640, plots=True,
                       project=str(RUNS), name='teste_' + tag, exist_ok=True)
    df = pd.DataFrame({'classe': [r.names[i] for i in r.box.ap_class_index],
                       'precisao': r.box.p, 'recall': r.box.r,
                       'mAP50': r.box.ap50, 'mAP50-95': r.box.ap})
    return r, df

r_s, pc_s = por_classe(pesos['yolo26s'], 'yolo26s')
r_n, pc_n = por_classe(pesos['yolo26n'], 'yolo26n')

print('yolo26s')
display(pc_s.round(4))
print('yolo26n')
display(pc_n.round(4))

In [ ]:
for nome in ['confusion_matrix_normalized.png', 'confusion_matrix.png']:
    f = Path(r_s.save_dir) / nome
    if f.exists():
        plt.figure(figsize=(8, 7))
        plt.imshow(plt.imread(f))
        plt.axis('off')
        plt.show()
        break

## 8. Salvar artefatos

In [ ]:
E = BASE / 'entrega_grupo'
shutil.rmtree(E, ignore_errors=True)
E.mkdir()

shutil.copy(pesos['yolo26s'], E / 'best_epi_grupo_yolo26s.pt')
shutil.copy(pesos['yolo26n'], E / 'best_epi_grupo_yolo26n.pt')
shutil.copy(OUT / 'data.yaml', E / 'data_epi_grupo.yaml')
tabela.to_csv(E / 'metricas_grupo.csv', index=False)
pc_s.to_csv(E / 'metricas_por_classe_teste_yolo26s.csv', index=False)
pc_n.to_csv(E / 'metricas_por_classe_teste_yolo26n.csv', index=False)
meta.drop(columns='classes').to_csv(E / 'split_por_grupo.csv', index=False)

shutil.make_archive(str(BASE / 'epi_resultados_grupo'), 'zip', E)
print(sorted(p.name for p in E.iterdir()))